# Leçon 11 - Protocole Agent-à-Agent (A2A)


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Qu'est-ce que le protocole A2A ?

Le **protocole Agent-to-Agent (A2A)** est une norme ouverte qui permet aux agents IA de communiquer,
de se découvrir mutuellement et de collaborer — même s'ils sont construits sur des frameworks différents ou hébergés
par des services différents.

Concepts clés :

- **Découverte** – Les agents publient une *Carte d'Agent* qui décrit leurs capacités, facilitant ainsi
  la tâche aux autres agents (ou aux orchestrateurs) pour trouver le spécialiste adéquat pour une tâche.
- **Passage de messages** – Les agents échangent des messages structurés via un protocole commun, de sorte qu’une
  requête d’un agent puisse être comprise et réalisée par un autre quel que soit son fonctionnement interne.
- **Cycle de vie de la tâche** – A2A définit des états tels que *soumis*, *en cours*, *terminé* et
  *échoué*, donnant à l’orchestrateur une visibilité complète sur l’avancement d’une tâche déléguée.

Dans cette leçon, nous simulons une collaboration de type A2A en connectant trois agents de voyage spécialisés
dans un flux de travail où chaque agent apporte son expertise et transmet les résultats au suivant.


## Création d'agents de voyage spécialisés


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Collaboration multi-agent via workflow

Nous connectons les trois agents dans un workflow séquentiel qui reflète la transmission de messages A2A :

1. **CurrencyExchangeAgent** reçoit la demande de l'utilisateur et produit des conseils sur les devises.
2. **ActivityPlannerAgent** reçoit le contexte enrichi et ajoute des recommandations d'activités.
3. **TravelManagerAgent** synthétise les deux entrées en un brief de voyage final.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Comprendre l'A2A en production

Dans un environnement de production, le protocole A2A ouvre des scénarios puissants entre services :

| Fonctionnalité | Description |
|---|---|
| **Interopérabilité inter-frameworks** | Un agent construit avec un framework peut déléguer des tâches à un agent construit avec un autre framework compatible A2A, permettant une véritable interopérabilité inter-organisationnelle. |
| **Limites des services** | Les agents peuvent vivre dans des microservices séparés, des régions cloud ou même différentes organisations tout en collaborant sans interruption. |
| **Découverte dynamique** | Un orchestrateur peut interroger un registre de cartes d’agent au moment de l’exécution pour trouver le spécialiste le mieux adapté à une sous-tâche donnée. |
| **Streaming & notifications push** | L'A2A prend en charge les Server-Sent Events (SSE) pour les mises à jour en temps réel des progrès et les notifications push pour les tâches longues. |

Le flux de travail que nous avons construit ci-dessus est une version simplifiée, en processus, de ce modèle. Dans un déploiement réel, chaque agent exposerait un point de terminaison HTTP, publierait une carte d’agent, et communiquerait via le protocole JSON-RPC A2A.


## Résumé

Dans cette leçon, vous avez appris :

1. **Ce qu’est le protocole A2A** — une norme ouverte pour la découverte, la messagerie et la gestion des tâches entre agents.
2. **Comment créer des agents spécialisés** — un agent de change monétaire, un agent planificateur d’activités, et un orchestrateur de gestion de voyages.
3. **Comment intégrer les agents dans un flux de travail** — en utilisant `WorkflowBuilder` pour modéliser la transmission séquentielle de messages entre agents.
4. **Comment fonctionne A2A en production** — permettant une collaboration inter-frameworks et inter-services avec une découverte dynamique et des mises à jour en continu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçons d’assurer la précision, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle humaine. Nous déclinons toute responsabilité en cas de malentendus ou d’interprétations erronées résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
